
# 03. Model Comparison

## 1. Objective

This notebook evaluates whether nonlinear models can
improve claim severity predictions beyond the Ridge
Regression baseline.

We begin with CatBoost, a gradient boosting algorithm
that supports categorical features natively.

Models will be evaluated using Mean Absolute Error (MAE)
on the same validation split established previously.

**Current benchmark:** Ridge Regression with
non-negative clipping (MAE: 1,288.44).


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

train = pd.read_csv(project_root / "data" / "raw" / "train.csv")

X = train.drop(columns=["id", "loss"])
y = train["loss"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

cat_cols = X_train.select_dtypes(include=["object", "str", "category"]).columns.tolist()

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("Categorical features:", len(cat_cols))

Training shape: (150654, 130)
Validation shape: (37664, 130)
Categorical features: 116



## 2. CatBoost Regression

CatBoost is a gradient boosting algorithm designed
to handle numerical and categorical features.

Unlike Ridge Regression, CatBoost can capture nonlinear
relationships and feature interactions without requiring
external One-Hot Encoding.

We train an initial model using the original features
and evaluate its performance using validation MAE.

The initial configuration will serve as a starting point
before any hyperparameter tuning.


In [2]:
from catboost import CatBoostRegressor

catboost_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    allow_writing_files=False,
    verbose=100,
)

catboost_model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50,
    use_best_model=True,
)

0:	learn: 1777.2328823	test: 1762.1270442	best: 1762.1270442 (0)	total: 449ms	remaining: 3m 44s
100:	learn: 1213.6003927	test: 1201.1136362	best: 1201.1136362 (100)	total: 36.7s	remaining: 2m 25s
200:	learn: 1178.4797574	test: 1170.7447602	best: 1170.7447602 (200)	total: 1m 15s	remaining: 1m 51s
300:	learn: 1160.7429918	test: 1157.8951119	best: 1157.8951119 (300)	total: 2m 1s	remaining: 1m 20s
400:	learn: 1148.9855173	test: 1150.9483254	best: 1150.9483254 (400)	total: 2m 40s	remaining: 39.6s
499:	learn: 1141.2839345	test: 1147.3101704	best: 1147.3101704 (499)	total: 3m 17s	remaining: 0us

bestTest = 1147.31017
bestIteration = 499



CatBoostRegressor(allow_writing_files=False, depth=6, eval_metric='MAE', iterations=500, learning_rate=0.05, loss_function='MAE', random_seed=42, verbose=100)

In [3]:
from sklearn.metrics import mean_absolute_error

catboost_pred = catboost_model.predict(X_valid)
catboost_mae = mean_absolute_error(y_valid, catboost_pred)

print(f"CatBoost MAE: {catboost_mae:,.2f}")
print(f"Best iteration: {catboost_model.best_iteration_}")
print(f"Negative predictions: {(catboost_pred < 0).sum()}")

CatBoost MAE: 1,147.31
Best iteration: 499
Negative predictions: 0



### 2.1 CatBoost Findings

CatBoost achieved a validation MAE of **1,147.31**, improving
upon the clipped Ridge baseline (1,288.44) by approximately
10.95%.

**Key Findings:**
- Training and validation MAE were closely aligned, with a gap of 6.03.
- No negative predictions were generated.
- The best iteration was 499, the final boosting iteration.

**Modeling Decision:**

CatBoost becomes the current performance benchmark.
Since validation MAE continued improving through the final
iteration, we will investigate whether additional boosting
iterations improve generalization.



### 2.2 Effect of Boosting Iterations

The initial CatBoost model achieved its best validation
MAE at the final iteration, suggesting that additional
boosting rounds may improve performance.

We increase the maximum number of iterations from 500
to 1,000 while keeping the remaining parameters fixed.

Early stopping is used to limit unnecessary training
if validation performance stops improving.


In [4]:
catboost_1000 = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    allow_writing_files=False,
    verbose=100,
    thread_count=2,
)

catboost_1000.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50,
    use_best_model=True,
)

0:	learn: 1777.2328823	test: 1762.1270442	best: 1762.1270442 (0)	total: 917ms	remaining: 15m 15s
100:	learn: 1213.6003927	test: 1201.1136362	best: 1201.1136362 (100)	total: 1m 25s	remaining: 12m 37s
200:	learn: 1178.4797574	test: 1170.7447602	best: 1170.7447602 (200)	total: 2m 54s	remaining: 11m 34s
300:	learn: 1160.7429918	test: 1157.8951119	best: 1157.8951119 (300)	total: 4m 31s	remaining: 10m 31s
400:	learn: 1148.9855173	test: 1150.9483254	best: 1150.9483254 (400)	total: 6m 3s	remaining: 9m 2s
500:	learn: 1141.2257956	test: 1147.2963687	best: 1147.2963687 (500)	total: 7m 33s	remaining: 7m 31s
600:	learn: 1134.6127410	test: 1144.2202685	best: 1144.2202685 (600)	total: 9m 8s	remaining: 6m 4s
700:	learn: 1129.8948283	test: 1142.3155220	best: 1142.3155220 (700)	total: 10m 41s	remaining: 4m 33s
800:	learn: 1125.9618313	test: 1140.9003635	best: 1140.9003635 (800)	total: 12m 11s	remaining: 3m 1s
900:	learn: 1122.4134952	test: 1139.6103859	best: 1139.6103859 (900)	total: 13m 43s	remaining: 

CatBoostRegressor(allow_writing_files=False, depth=6, eval_metric='MAE', iterations=1000, learning_rate=0.05, loss_function='MAE', random_seed=42, thread_count=2, verbose=100)

In [5]:
catboost_1000_pred = catboost_1000.predict(X_valid)

catboost_1000_mae = mean_absolute_error(y_valid, catboost_1000_pred)

print(f"CatBoost 500 MAE: {catboost_mae:,.2f}")
print(f"CatBoost 1000 MAE: {catboost_1000_mae:,.2f}")
print(f"Negative predictions: {(catboost_1000_pred < 0).sum()}")

CatBoost 500 MAE: 1,147.31
CatBoost 1000 MAE: 1,138.68
Negative predictions: 0



### 2.2 Iteration Experiment Findings

Increasing CatBoost iterations from 500 to 1,000
reduced validation MAE from **1,147.31** to **1,138.68**,
an improvement of approximately 0.75%.

**Key Findings:**
- The best iteration was 997, near the training limit.
- No negative predictions were generated.
- The training-validation gap increased slightly.
- Validation improvements became smaller in later iterations.

**Modeling Decision:**

Retain the 1,000-iteration CatBoost as the current
benchmark. Further tuning will be guided by
cross-validation rather than repeatedly extending
the number of iterations.



## 3. XGBoost Regression

XGBoost is a gradient boosting algorithm capable of
learning nonlinear relationships and feature interactions.

Unlike our CatBoost implementation, this experiment uses
One-Hot Encoding for categorical features.

We train an initial XGBoost model with a fixed configuration
and compare its validation MAE against the CatBoost benchmark.

The same training and validation split is preserved to
ensure a consistent evaluation.


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", X_train.select_dtypes(include="number").columns),
    ]
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessing", xgb_preprocessor),
        (
            "model",
            XGBRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=6,
                objective="reg:absoluteerror",
                eval_metric="mae",
                tree_method="hist",
                n_jobs=2,
                random_state=42,
            ),
        ),
    ]
)

In [7]:
xgb_pipeline.fit(X_train, y_train)

xgb_pred = xgb_pipeline.predict(X_valid)

xgb_mae = mean_absolute_error(y_valid, xgb_pred)

print(f"XGBoost Validation MAE: {xgb_mae:,.2f}")
print(f"Negative predictions: {(xgb_pred < 0).sum()}")

XGBoost Validation MAE: 1,145.58
Negative predictions: 0



### 3.1 XGBoost Findings

XGBoost achieved a validation MAE of **1,145.58**, outperforming
the Ridge baseline but remaining slightly behind the best
CatBoost model (1,138.68).

**Key Findings:**
- XGBoost improved substantially over the linear baseline.
- No negative predictions were generated.
- CatBoost currently leads by 6.90 MAE points.

**Next Step:**

Assess the training-validation gap before drawing conclusions
about model generalization.


In [8]:
xgb_train_pred = xgb_pipeline.predict(X_train)

xgb_train_mae = mean_absolute_error(y_train, xgb_train_pred)

print(f"Training MAE: {xgb_train_mae:,.2f}")
print(f"Validation MAE: {xgb_mae:,.2f}")
print(f"Generalization gap: {xgb_mae - xgb_train_mae:,.2f}")

Training MAE: 1,099.84
Validation MAE: 1,145.58
Generalization gap: 45.74



### 3.2 XGBoost Generalization Assessment

XGBoost achieved a training MAE of **1,099.84** and a
validation MAE of **1,145.58**, resulting in a
generalization gap of **45.74**.

**Key Findings:**
- XGBoost achieved lower training error than CatBoost.
- Its validation performance remained slightly weaker.
- The larger generalization gap suggests a greater
  tendency toward overfitting.

**Modeling Decision:**

Retain CatBoost as the current benchmark.
Use cross-validation to assess model stability before
further hyperparameter tuning.



## 4. Cross-Validation

A single validation split may not fully represent model
generalization performance.

We use 3-fold cross-validation on the training subset
to assess the stability of our candidate models across
different data partitions.

All preprocessing and model fitting will be performed
independently within each fold to prevent data leakage.

We will compare mean MAE and its variability across folds.


In [9]:
from sklearn.model_selection import KFold

cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train), start=1):
    print(f"Fold {fold}: Train={len(train_idx):,}, Validation={len(val_idx):,}")

Fold 1: Train=100,436, Validation=50,218
Fold 2: Train=100,436, Validation=50,218
Fold 3: Train=100,436, Validation=50,218



### 4.1 CatBoost Cross-Validation

We evaluate the 1,000-iteration CatBoost configuration
using three training folds.

Each model is trained from scratch, and performance is
measured on its corresponding held-out fold.

We keep the model configuration fixed to assess
validation stability without additional tuning.


In [10]:
train_idx, val_idx = next(cv.split(X_train))

X_fold_train = X_train.iloc[train_idx]
X_fold_valid = X_train.iloc[val_idx]

y_fold_train = y_train.iloc[train_idx]
y_fold_valid = y_train.iloc[val_idx]

cv_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="MAE",
    random_seed=42,
    thread_count=2,
    allow_writing_files=False,
    verbose=False,
)

cv_model.fit(
    X_fold_train,
    y_fold_train,
    cat_features=cat_cols,
)

fold_pred = cv_model.predict(X_fold_valid)
fold_mae = mean_absolute_error(y_fold_valid, fold_pred)

print(f"Fold 1 MAE: {fold_mae:,.2f}")

Fold 1 MAE: 1,159.84


In [11]:
import gc

import numpy as np

cv_maes = [fold_mae]

del cv_model
gc.collect()

for fold_num, (train_idx, val_idx) in enumerate(cv.split(X_train), start=1):
    if fold_num == 1:
        continue

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        loss_function="MAE",
        random_seed=42,
        thread_count=2,
        allow_writing_files=False,
        verbose=False,
    )

    model.fit(
        X_train.iloc[train_idx],
        y_train.iloc[train_idx],
        cat_features=cat_cols,
    )

    predictions = model.predict(X_train.iloc[val_idx])
    mae = mean_absolute_error(y_train.iloc[val_idx], predictions)

    cv_maes.append(mae)
    print(f"Fold {fold_num} MAE: {mae:,.2f}")

    del model
    gc.collect()

print(f"Mean CV MAE: {np.mean(cv_maes):,.2f}")
print(f"CV Standard Deviation: {np.std(cv_maes):,.2f}")

Fold 2 MAE: 1,146.62
Fold 3 MAE: 1,156.71
Mean CV MAE: 1,154.39
CV Standard Deviation: 5.64



### 4.2 CatBoost Cross-Validation Findings

The 3-fold cross-validation produced a mean MAE of
**1,154.39**, with a standard deviation of **5.64**.

**Key Findings:**
- Validation MAE ranged from 1,146.62 to 1,159.84.
- The relatively low standard deviation suggests
  consistent performance across the three folds.
- The mean CV MAE was higher than the original
  holdout validation MAE of 1,138.68.

**Interpretation:**

CatBoost demonstrated reasonably stable performance
across different training and validation partitions.
However, cross-validation results for XGBoost are needed
before making a more reliable model comparison.



### 4.3 XGBoost Cross-Validation

We evaluate the XGBoost pipeline using the same three
cross-validation folds as CatBoost.

The preprocessing pipeline is fitted independently
within each fold to prevent data leakage.

We compare mean validation MAE and standard deviation
to assess relative performance and stability.


In [12]:
from sklearn.model_selection import cross_val_score

xgb_cv_scores = cross_val_score(
    xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=1,
    verbose=2,
)

xgb_cv_maes = -xgb_cv_scores

for fold, mae in enumerate(xgb_cv_maes, start=1):
    print(f"Fold {fold} MAE: {mae:,.2f}")

print(f"Mean CV MAE: {np.mean(xgb_cv_maes):,.2f}")
print(f"CV Standard Deviation: {np.std(xgb_cv_maes):,.2f}")

[CV] END .................................................... total time=  11.2s
[CV] END .................................................... total time=  13.9s
[CV] END .................................................... total time=  11.4s
Fold 1 MAE: 1,167.98
Fold 2 MAE: 1,154.28
Fold 3 MAE: 1,164.56
Mean CV MAE: 1,162.28
CV Standard Deviation: 5.82


[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:   36.8s finished



### 4.4 Cross-Validation Comparison and Model Selection

CatBoost achieved a mean cross-validation MAE of **1,154.39**
(SD: 5.64), compared with **1,162.28** (SD: 5.82) for XGBoost.

**Key Findings:**
- CatBoost outperformed XGBoost in all three folds.
- CatBoost achieved a 7.89-point reduction in mean MAE,
  representing approximately 0.68% improvement.
- Both models showed relatively consistent performance
  across validation folds.
- XGBoost required substantially less training time.

**Model Selection:**

CatBoost is selected as the preferred model based on
its lower cross-validation MAE and consistent advantage
across all three folds.

The performance difference is modest, so this result
should not be interpreted as proof of universal superiority.
The selected model will be used for the final prediction
and submission workflow.
